### Convert SMILES Strings into completed 'UMD' files
UMD: Universal Molecular Description files. This is a precursor to the more compressed and faster-to-traverse MF (Universal Molecular Format) files

In [1]:
import os,sys
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from openff.toolkit import Molecule
from openff.toolkit.utils.toolkits import AmberToolsToolkitWrapper
from openff.toolkit.utils.exceptions import UndefinedStereochemistryError
sys.path.append("/home/venkata/python/python_libraries/CADD/")
from General import *

KeyboardInterrupt: 

In [ ]:
'''
props=AllChem.MMFFGetMoleculeProperties(m1h)
for i in range(m1h.GetNumAtoms()):
    charge = props.GetMMFFPartialCharge(i)
    print(charge)
'''
print("OpenFF Molecule parameterizer loading")
HYBRIDIZATION={"S": 0, "SP": 1, "SP2": 2, "SP3": 3, "SP3D": 4, "SP3D2": 5}
ELEMENTS={1: "H", 3: "Li", 4: "Be", 5: "B", 6: "C", 7: "N", 8: "O", 9: "F",
          12: "Mg", 14: "Si", 15: "P", 16: "S", 17: "Cl", 26: "Fe", 35: "Br",
          53: "I", 46: "Pd", 78: "Pt", 34: "Se", 30: "Zn"}
BOND_ORDERS={rdkit.Chem.rdchem.BondType.SINGLE: 2, rdkit.Chem.rdchem.BondType.DOUBLE: 4, rdkit.Chem.rdchem.BondType.AROMATIC: 3, rdkit.Chem.rdchem.BondType.TRIPLE: 6}

In [ ]:
def openff_charge(mol,method="am1bcc"):
    off_mol = Molecule.from_rdkit(mol)
    off_mol.assign_partial_charges(partial_charge_method=method)
    rdkit_mol = off_mol.to_rdkit()
    return rdkit_mol

def rdkit_mmff94(mol):
    props=AllChem.MMFFGetMoleculeProperties(mol)
    for i,a in enumerate(mol.GetAtoms()):
        charge = props.GetMMFFPartialCharge(i)
        a.SetDoubleProp("PartialCharge",charge)
    return mol

def no_charges(mol): return mol

failmol=None
def processMolecule(smistr, name=None, charge_method="am1bcc"):
    global failmol
    namerep=smistr if name is None else name
    mol=Chem.MolFromSmiles(smistr)
    if mol is None: raise ValueError(f"RDKit could not process the SMILES String '{smistr}'")

    mol_h=AllChem.AddHs(mol)
    eret=AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv2())
    if eret!=0:
        if eret==-1:
            failmol=mol
            raise ValueError("Valency satisfaction failed. You can check the global 'failmol' object to see if the molecule has obvious flaws")
        else:
            print("WARN: Molecule has multiple 3D representations. One will be chosen at random")
    AllChem.AssignStereochemistryFrom3D(mol_h)
    try: minret=AllChem.MMFFOptimizeMolecule(mol_h)
    except UndefinedStereochemistryError: pass
    if minret==-1:
        print(f"WARN: Minimization for '{namerep}' failed to run (This is NOT a convergence issue; the forcefield failed to setup)")
    elif minret>0:
        print(f"WARN: Minimization for '{namerep}' did not converge. This might be OK")
    if type(charge_method)==str:
        charged_mol = openff_charge(mol_h,charge_method)
    else:
        try: charged_mol =charge_method(mol_h)
        except:
            try: charged_mol = openff_charge(mol_h,str(charge_method))
            except: raise ValueError(f"Failed to add charges. Method '{str(charge_method)}' failed to be called as a function and as a known charge-method for OpenFF")
        
    if charged_mol is None: raise ValueError(f"Charge calculation failed for '{namerep}'")
    ready_mol = Chem.Kekulize(charged_mol)
    if ready_mol is None:
        print(f"WARN: Molecule Kekulization faced errors for '{namerep}'. Bond orders may be wrong")
        ready_mol=charged_mol
        
    return ready_mol

In [ ]:
def raw_dump(smiles,outf,molname=None):
    ready_mol = processMolecule(str(smiles),name=molname, charge_method="gasteiger")
    if molname is None: molname="Unnamed"

    madefile=False
    if type(outf)==str:
        outf=open(outf,"w")
        madefile=True
    outf.write("START\n")
    outf.write("; Name\n")
    outf.write(str(molname)+"\n")
    outf.write("; SMILES String\n")
    outf.write(str(smiles)+"\n")
    outf.write("; # Atoms\n")
    outf.write(str(ready_mol.GetNumAtoms())+"\n")
    outf.write("; # Bonds\n")
    outf.write(str(ready_mol.GetNumBonds())+"\n")
    
    outf.write("; Atoms\n")
    outf.write("; Index Element X Y Z Q Hyb Aromatic \n")
    
    conf = ready_mol.GetConformer()
    for ai,a in enumerate(ready_mol.GetAtoms()):
        wstr=str(ai)+" "+ELEMENTS[a.GetAtomicNum()]+" "
        pos = conf.GetAtomPosition(ai)
        wstr+=str(pos.x)+" "+str(pos.y)+" "+str(pos.z)+" "
        q=round(a.GetDoubleProp("PartialCharge"),3)
        wstr+=str(q)+" "
        wstr+=str(HYBRIDIZATION[str(a.GetHybridization())])+" "
        wstr+=("1" if a.GetIsAromatic() else "0")+"\n"
        outf.write(wstr)

    outf.write("; Bonds\n")
    outf.write("; Index At1 At2 2*Order\n")
    for bi,b in enumerate(ready_mol.GetBonds()):
        wstr=str(bi)+" "+str(b.GetBeginAtomIdx())+" "+str(b.GetEndAtomIdx())+" "
        wstr+=str(BOND_ORDERS[b.GetBondType()])+"\n"
        outf.write(wstr)
    outf.write("END\n")
    if madefile: outf.close()

In [ ]:
smiload=SequentialSMILESLoader("enamine_clean.smi",attach_names=True)
outf=open("EnAmine_processed_shuffled.umd","w")
while not smiload.ended:
    smis,names=smiload.getNext(2048,as_smiles=True)
    for i in range(len(smis)):
        try:
            raw_dump(smis[i],outf,names[i])
        except UndefinedStereochemistryError: continue
        except ValueError: continue
    print("Batch complete")
outf.close()